# Solar Installation Duration — Model Analysis
**Goal:** Push R² above 0.55 through feature engineering and model selection.

Sections:
1. Setup & Data
2. Exploratory Data Analysis (EDA)
3. Feature Engineering
4. Model Comparison
5. Residual Analysis
6. Best Model Summary

In [ ]:
# Install dependencies
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'seaborn'])
print('Dependencies ready.')

In [ ]:
import io, os, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import r2_score, mean_squared_error

pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.family'] = 'Arial'
RANDOM_STATE = 42
print('Imports done.')

## 1. Load Data
Full Working Data sheet — roof mounts only, 264 jobs, 30 columns. Drive time and salary hours converted to decimal hours.

In [ ]:
DATA_PATH = 'data/solar_data_full.csv'

if not os.path.exists(DATA_PATH):
    from google.colab import files
    print('File not found — uploading now...')
    uploaded = files.upload()  # select solar_data_full.csv
    os.makedirs('data', exist_ok=True)
    import shutil
    shutil.move(list(uploaded.keys())[0], DATA_PATH)

df = pd.read_csv(DATA_PATH)

for col in ['Tilt', 'Azimuth', 'Panel QTY', 'System Rating (kW DC)',
            '# of Arrays', '# of reinforcement', '# of Stories',
            'Module Length', 'Module Width', 'Module Weight',
            'Estimated Total # of People on Site', 'Estimated Total Direct Time',
            'Total # of Days on Site', 'Estimated Salary Hours']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print('Shape:', df.shape)
df.head()

In [ ]:
print('Columns:')
for c in df.columns:
    print(' -', c)
print('\nNull counts:')
print(df.isnull().sum())
print('\nDescriptive stats:')
df.describe()

## 2. Exploratory Data Analysis

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['Estimated Total Direct Time'].dropna(), bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Install Duration (hours)')
axes[0].set_xlabel('Hours')
axes[0].set_ylabel('Count')
axes[1].hist(np.log1p(df['Estimated Total Direct Time'].dropna()), bins=30, color='steelblue', edgecolor='white')
axes[1].set_title('Log Distribution of Install Duration')
axes[1].set_xlabel('Log(Hours + 1)')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.show()
print('Skewness:', round(df['Estimated Total Direct Time'].skew(), 3))
print('Outliers (>2 std):', (np.abs(df['Estimated Total Direct Time'] - df['Estimated Total Direct Time'].mean()) > 2 * df['Estimated Total Direct Time'].std()).sum())

In [ ]:
# Numeric correlations with target
numeric_cols = ['Panel QTY', 'Tilt', 'Azimuth', 'System Rating (kW DC)',
                '# of Arrays', '# of reinforcement', '# of Stories',
                'Module Length', 'Module Width', 'Module Weight',
                'Estimated Total # of People on Site']
corrs = df[numeric_cols + ['Estimated Total Direct Time']].corr()['Estimated Total Direct Time'].drop('Estimated Total Direct Time')
print('Correlations with Install Duration:')
print(corrs.sort_values(ascending=False).round(4))

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(
    df[numeric_cols + ['Estimated Total Direct Time']].corr(),
    annot=True, fmt='.2f', cmap='RdYlGn', center=0,
    vmin=-1, vmax=1, linewidths=0.5, linecolor='white',
    ax=ax, annot_kws={'size': 8}
)
ax.set_title('Correlation Matrix — Numeric Variables', fontsize=13, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Categorical breakdowns — box plots
cat_cols = ['Install Season', 'Roof Type', 'Inverter Manufacturer',
            'Truss / Rafter', 'Reinforcements', 'Squirrel Screen',
            'Rough Electrical Inspection', 'Portrait / Landscape']

fig, axes = plt.subplots(4, 2, figsize=(14, 22))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    groups = [grp['Estimated Total Direct Time'].dropna().values
              for _, grp in df.groupby(col)]
    labels = [str(k) for k in df.groupby(col).groups.keys()]
    counts = [df[df[col] == k].shape[0] for k in df.groupby(col).groups.keys()]

    bp = axes[i].boxplot(groups, patch_artist=True, medianprops=dict(color='red', linewidth=2))
    for patch in bp['boxes']:
        patch.set_facecolor('steelblue')
        patch.set_alpha(0.6)

    axes[i].set_xticks(range(1, len(labels) + 1))
    axes[i].set_xticklabels(
        [f'{l}\n(n={c})' for l, c in zip(labels, counts)],
        fontsize=8, rotation=15, ha='right'
    )
    axes[i].set_title(f'Duration by {col}', fontsize=11, fontweight='bold')
    axes[i].set_ylabel('Hours')
    axes[i].grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Panel QTY vs Duration coloured by key categoricals
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, ['Install Season', 'Roof Type', 'Reinforcements']):
    for label, group in df.groupby(col):
        ax.scatter(group['Panel QTY'], group['Estimated Total Direct Time'],
                   label=label, alpha=0.5, s=30)
    ax.set_title(f'Panel QTY vs Duration\n(coloured by {col})')
    ax.set_xlabel('Panel QTY')
    ax.set_ylabel('Duration (hours)')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 3. Feature Engineering
Linear regression assumes straight-line relationships. We can create new features to capture curves and interactions between variables.

In [ ]:
df_eng = df.copy()

# Panels per array — density/complexity per section
df_eng['Panels per Array'] = df_eng['Panel QTY'] / df_eng['# of Arrays'].replace(0, 1)

# Panels per worker — individual workload
df_eng['Panels per Worker'] = df_eng['Panel QTY'] / df_eng['Estimated Total # of People on Site'].replace(0, 1)

# Panel QTY squared — non-linear scaling
df_eng['Panel QTY^2'] = df_eng['Panel QTY'] ** 2

# Interaction: panels x arrays
df_eng['Panel QTY x Arrays'] = df_eng['Panel QTY'] * df_eng['# of Arrays']

# System size per worker
df_eng['kW per Worker'] = df_eng['System Rating (kW DC)'] / df_eng['Estimated Total # of People on Site'].replace(0, 1)

eng_features = ['Panels per Array', 'Panels per Worker', 'Panel QTY^2', 'Panel QTY x Arrays', 'kW per Worker']
corrs_eng = df_eng[eng_features + ['Estimated Total Direct Time']].corr()['Estimated Total Direct Time'].drop('Estimated Total Direct Time')
print('Engineered feature correlations with duration:')
print(corrs_eng.sort_values(ascending=False).round(4))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(eng_features):
    axes[i].scatter(df_eng[col], df_eng['Estimated Total Direct Time'], alpha=0.4, color='steelblue')
    axes[i].set_title(f'{col} vs Duration')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Duration (hours)')

axes[len(eng_features)].axis('off')
plt.tight_layout()
plt.show()

## 4. Model Comparison

- **Linear Regression** — baseline
- **Ridge** — linear with regularization (reduces overfitting)
- **Lasso** — like Ridge, also removes weak features automatically
- **Random Forest** — ensemble of decision trees, handles non-linear relationships
- **Gradient Boosting** — sequential trees, typically strongest on tabular data

In [ ]:
def build_features(dataframe, include_engineered=False):
    X = dataframe[['Panel QTY', 'Estimated Total # of People on Site', '# of Arrays',
                    'System Rating (kW DC)', '# of Stories']].copy()
    X = pd.concat([X,
        pd.get_dummies(dataframe['Install Season'], prefix='Season', drop_first=True),
        pd.get_dummies(dataframe['Roof Type'], prefix='Roof', drop_first=True),
        pd.get_dummies(dataframe['Reinforcements'], prefix='Reinforce', drop_first=True),
        pd.get_dummies(dataframe['Squirrel Screen'], prefix='Squirrel', drop_first=True),
    ], axis=1)
    if include_engineered:
        X['Panels per Array']    = dataframe['Panel QTY'] / dataframe['# of Arrays'].replace(0, 1)
        X['Panels per Worker']   = dataframe['Panel QTY'] / dataframe['Estimated Total # of People on Site'].replace(0, 1)
        X['Panel QTY^2']         = dataframe['Panel QTY'] ** 2
        X['Panel QTY x Arrays']  = dataframe['Panel QTY'] * dataframe['# of Arrays']
        X['kW per Worker']       = dataframe['System Rating (kW DC)'] / dataframe['Estimated Total # of People on Site'].replace(0, 1)
    return X.apply(pd.to_numeric, errors='coerce')

y = df_eng['Estimated Total Direct Time']
X_base = build_features(df_eng, include_engineered=False)
X_eng  = build_features(df_eng, include_engineered=True)

mask_base = X_base.notna().all(axis=1) & y.notna()
mask_eng  = X_eng.notna().all(axis=1) & y.notna()

print('Base features:', list(X_base.columns))
print('\nEngineered features added:', [c for c in X_eng.columns if c not in X_base.columns])

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {
    'Linear Regression':  LinearRegression(),
    'Ridge':              Ridge(alpha=1.0),
    'Lasso':              Lasso(alpha=1.0),
    'Random Forest':      RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE),
    'Gradient Boosting':  GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_STATE),
}

results = []
for name, m in models.items():
    for X_set, label, mask in [
        (X_base, 'Base', mask_base),
        (X_eng,  'Engineered', mask_eng)
    ]:
        X_c, y_c = X_set[mask], y[mask]
        scores = cross_val_score(m, X_c, y_c, cv=kf, scoring='r2')
        results.append({'Model': name, 'Features': label,
                        'Mean R²': round(scores.mean(), 4),
                        'Std R²': round(scores.std(), 4),
                        'Min R²': round(scores.min(), 4),
                        'Max R²': round(scores.max(), 4)})

results_df = pd.DataFrame(results).sort_values('Mean R²', ascending=False)
results_df

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
pivot = results_df.drop_duplicates(subset=['Model', 'Features']).pivot_table(index='Model', columns='Features', values='Mean R²', aggfunc='mean')
pivot.plot(kind='bar', ax=ax, color=['steelblue', 'darkorange'], edgecolor='white', width=0.6)
ax.set_title('Model R² Comparison — Base vs Engineered Features', fontsize=14, fontweight='bold')
ax.set_ylabel('Mean R² (5-fold CV)')
ax.set_xlabel('')
ax.axhline(0.5535, color='red', linestyle='--', linewidth=1.5, label='Original baseline (0.5535)')
ax.legend()
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

## 5. Residual Analysis
Fit the best model on a train/test split and examine where predictions go wrong.

In [ ]:
best_name = results_df.iloc[0]['Model']
best_features = results_df.iloc[0]['Features']
print(f'Best model: {best_name} ({best_features} features)')

X_best = X_eng[mask_eng] if best_features == 'Engineered' else X_base[mask_base]
y_best = y[mask_eng if best_features == 'Engineered' else mask_base]

X_train, X_test, y_train, y_test = train_test_split(X_best, y_best, test_size=0.2, random_state=RANDOM_STATE)

best_model = models[best_name]
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print(f'Test R²:   {r2_score(y_test, y_pred):.4f}')
print(f'Test RMSE: {mean_squared_error(y_test, y_pred)**0.5:.2f} hours')

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].scatter(y_pred, y_test, alpha=0.5, color='steelblue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=1.5)
axes[0].set_title('Predicted vs Actual')
axes[0].set_xlabel('Predicted Hours')
axes[0].set_ylabel('Actual Hours')

axes[1].scatter(y_pred, residuals, alpha=0.5, color='steelblue')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_title('Residuals vs Predicted')
axes[1].set_xlabel('Predicted Hours')
axes[1].set_ylabel('Residual')

axes[2].hist(residuals, bins=25, color='steelblue', edgecolor='white')
axes[2].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[2].set_title('Residual Distribution')
axes[2].set_xlabel('Residual (hours)')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=X_best.columns).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(10, 7))
    importances.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'Feature Importance — {best_name}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()
else:
    print('Feature importance not available for linear models.')

## 6. Best Model Summary

In [ ]:
print('='*55)
print('BEST MODEL SUMMARY')
print('='*55)
print(f'\nModel:          {best_name}')
print(f'Feature set:    {best_features}')
print(f'Test R²:        {r2_score(y_test, y_pred):.4f}')
print(f'Test RMSE:      {mean_squared_error(y_test, y_pred)**0.5:.2f} hours')
print(f'\nOriginal baseline R²: 0.5535')
print(f'Improvement:          +{round(r2_score(y_test, y_pred) - 0.5535, 4)}')
print('\nTop 5 models by cross-validated R²:')
print(results_df[['Model', 'Features', 'Mean R²', 'Std R²']].head(5).to_string(index=False))
print('\nWorst predictions (largest residuals):')
residual_df = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred.round(2),
    'Error (hrs)': residuals.values.round(2),
    'Error (%)': (residuals.values / y_test.values * 100).round(1)
})
print(residual_df.reindex(residual_df['Error (hrs)'].abs().sort_values(ascending=False).index).head(10).to_string(index=False))

## 7. Production Model — Interaction Term Validation
Compare the production model (Panel QTY + Crew + Arrays + Season) against the same model
plus a **Panel QTY × Crew Size** interaction. Averaged over 10 random seeds to reduce split noise.

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
import numpy as np

TARGET = 'Estimated Total Direct Time'
SEASON = 'Install Season'

def build_production_features(dataframe, interaction=False):
    """Mirrors the production model.py feature set."""
    X = dataframe[['Panel QTY', 'Estimated Total # of People on Site', '# of Arrays']].copy()
    X = pd.concat([
        X,
        pd.get_dummies(dataframe[SEASON], prefix=SEASON, drop_first=True)
    ], axis=1).apply(pd.to_numeric, errors='coerce')
    if interaction:
        X['Panel QTY x Crew Size'] = X['Panel QTY'] * X['Estimated Total # of People on Site']
    return X

y_prod = pd.to_numeric(df[TARGET], errors='coerce')
X_prod_base = build_production_features(df, interaction=False)
X_prod_int  = build_production_features(df, interaction=True)

mask_b = X_prod_base.notna().all(axis=1) & y_prod.notna()
mask_i = X_prod_int.notna().all(axis=1) & y_prod.notna()

n_seeds = 10
base_scores, int_scores = [], []

for seed in range(n_seeds):
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    base_scores.extend(cross_val_score(LinearRegression(), X_prod_base[mask_b], y_prod[mask_b], cv=kf, scoring='r2'))
    int_scores.extend(cross_val_score(LinearRegression(), X_prod_int[mask_i],  y_prod[mask_i],  cv=kf, scoring='r2'))

print('5-fold CV R² averaged over 10 random seeds (50 folds total):')
print(f'  Base model (no interaction): {np.mean(base_scores):.4f}  ±{np.std(base_scores):.4f}')
print(f'  + Panel x Crew interaction:  {np.mean(int_scores):.4f}  ±{np.std(int_scores):.4f}')
print(f'  Gain: +{np.mean(int_scores) - np.mean(base_scores):.4f}')

# MAE comparison
from sklearn.model_selection import cross_val_score
mae_base = -cross_val_score(LinearRegression(), X_prod_base[mask_b], y_prod[mask_b],
                             cv=KFold(5, shuffle=True, random_state=42), scoring='neg_mean_absolute_error')
mae_int  = -cross_val_score(LinearRegression(), X_prod_int[mask_i], y_prod[mask_i],
                             cv=KFold(5, shuffle=True, random_state=42), scoring='neg_mean_absolute_error')
mean_baseline = y_prod[mask_b].mean()
print(f'\nMAE baseline (predict mean): {mean_baseline:.1f} hrs')
print(f'MAE base model: {mae_base.mean():.2f} hrs  ({(1 - mae_base.mean()/mean_baseline)*100:.1f}% below baseline)')
print(f'MAE + interaction: {mae_int.mean():.2f} hrs  ({(1 - mae_int.mean()/mean_baseline)*100:.1f}% below baseline)')

### Production Model Diagnostics
Residual plots for the linear regression + interaction term on a held-out 20% test set.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import scipy.stats as stats

X_d = X_prod_int[mask_i]
y_d = y_prod[mask_i]

X_train, X_test, y_train, y_test = train_test_split(
    X_d, y_d, test_size=0.2, random_state=42
)

prod_model = LinearRegression()
prod_model.fit(X_train, y_train)
y_pred = prod_model.predict(X_test)
residuals = y_test - y_pred

print(f'Test R²:   {r2_score(y_test, y_pred):.4f}')
print(f'Test RMSE: {mean_squared_error(y_test, y_pred)**0.5:.2f} hrs')
print(f'Test MAE:  {np.abs(residuals).mean():.2f} hrs')
print()
print('Coefficients:')
for name, coef in zip(X_d.columns, prod_model.coef_):
    print(f'  {name:<45} {coef:+.4f}')
print(f'  Intercept: {prod_model.intercept_:.4f}')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Predicted vs Actual
axes[0, 0].scatter(y_pred, y_test, alpha=0.5, color='steelblue', edgecolors='white', linewidth=0.3)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0, 0].plot(lims, lims, 'r--', linewidth=1.5)
axes[0, 0].set_title('Predicted vs Actual', fontweight='bold')
axes[0, 0].set_xlabel('Predicted (hrs)')
axes[0, 0].set_ylabel('Actual (hrs)')

# Residuals vs Predicted
axes[0, 1].scatter(y_pred, residuals, alpha=0.5, color='steelblue', edgecolors='white', linewidth=0.3)
axes[0, 1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0, 1].set_title('Residuals vs Predicted', fontweight='bold')
axes[0, 1].set_xlabel('Predicted (hrs)')
axes[0, 1].set_ylabel('Residual (hrs)')

# Residual distribution
axes[1, 0].hist(residuals, bins=20, color='steelblue', edgecolor='white')
axes[1, 0].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1, 0].set_title('Residual Distribution', fontweight='bold')
axes[1, 0].set_xlabel('Residual (hrs)')
axes[1, 0].set_ylabel('Count')

# Q-Q plot
(osm, osr), (slope, intercept, r) = stats.probplot(residuals, dist='norm')
axes[1, 1].scatter(osm, osr, alpha=0.5, color='steelblue', s=20)
axes[1, 1].plot(osm, slope * np.array(osm) + intercept, 'r--', linewidth=1.5)
axes[1, 1].set_title('Q-Q Plot (Normality of Residuals)', fontweight='bold')
axes[1, 1].set_xlabel('Theoretical Quantiles')
axes[1, 1].set_ylabel('Sample Quantiles')

plt.suptitle('Production Model Diagnostics — Linear Regression + Panel×Crew Interaction',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()